In [67]:
import pandas as pd
import re
import unicodedata
import getpass
user = getpass.getuser()

lawyers = pd.read_csv(f"/Users/{user}/Final_Lawyer_Git July10/Data/Introduction/aba_county_lawyers.csv")

pop = pd.read_csv(
    f"/Users/{user}/Final_Lawyer_Git July10/Data/Population Data/County Population/co-est2019-alldata.csv",
    encoding="cp1252",
    dtype={"COUNTY": str}
)

pop = pop[pop["COUNTY"] != "000"].copy()


def name_key(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(x for x in value if not unicodedata.combining(x))
    return re.sub(r"[^a-z0-9]", "", value.lower())


def clean_county(state, county):
    separate_city_county = {
        ("Maryland", "Baltimore County"): "Baltimore County",
        ("Maryland", "Baltimore city"): "Baltimore City",
        ("Missouri", "St. Louis County"): "St. Louis County",
        ("Missouri", "St. Louis city"): "St. Louis City",
        ("Virginia", "Fairfax County"): "Fairfax County",
        ("Virginia", "Fairfax city"): "Fairfax City",
        ("Virginia", "Franklin County"): "Franklin County",
        ("Virginia", "Franklin city"): "Franklin City",
        ("Virginia", "Richmond County"): "Richmond County",
        ("Virginia", "Richmond city"): "Richmond City",
        ("Virginia", "Roanoke County"): "Roanoke County",
        ("Virginia", "Roanoke city"): "Roanoke City",
    }

    if (state, county) in separate_city_county:
        return separate_city_county[(state, county)]

    endings = [
        " City and Borough",
        " Census Area",
        " Municipality",
        " Borough",
        " Parish",
        " County",
        " city",
    ]

    for ending in endings:
        if county.endswith(ending):
            return county[:-len(ending)]

    return county


pop["merge_county"] = [
    clean_county(state, county)
    for state, county in zip(pop["STNAME"], pop["CTYNAME"])
]

fairfax_pop = pop.loc[
    (pop["STNAME"] == "Virginia")
    & (pop["CTYNAME"].isin(["Fairfax County", "Fairfax city"])),
    "POPESTIMATE2019"
].sum()

fairfax_row = pd.DataFrame({
    "STNAME": ["Virginia"],
    "merge_county": ["Fairfax City and County"],
    "POPESTIMATE2019": [fairfax_pop]
})

pop = pd.concat([pop, fairfax_row], ignore_index=True)

name_fixes = {
    ("North Carolina", "Guilford-High Pt."): "Guilford",
    ("Virginia", "Charlottesvlile"): "Charlottesville",
    ("Missouri", "St. Louis"): "St. Louis County",
}

lawyers["merge_county"] = [
    name_fixes.get((state, county), county)
    for state, county in zip(lawyers["State"], lawyers["County"])
]

lawyers["merge_key"] = (
    lawyers["State"].map(name_key)
    + lawyers["merge_county"].map(name_key)
)

pop["merge_key"] = (
    pop["STNAME"].map(name_key)
    + pop["merge_county"].map(name_key)
)

combined = lawyers.merge(
    pop[["merge_key", "POPESTIMATE2019"]],
    on="merge_key",
    how="left",
    validate="many_to_one"
)

combined = combined.rename(columns={"POPESTIMATE2019": "Pop2019"})
combined = combined[["State", "County", "Lawyers", "Pop2019"]]

print("Missing populations:", combined["Pop2019"].isna().sum())
print(combined[combined["Pop2019"].isna()])
combined.to_csv(
    f"/Users/{user}/Final_Lawyer_Git July10/Data/Introduction/aba_county_lawyers_intro.csv",
    index=False,
)


Missing populations: 1
     State   County  Lawyers  Pop2019
69  Alaska  Chugach        2      NaN


In [68]:
lawyers = pd.read_csv(f"/Users/{user}/Final_Lawyer_Git July10/Data/Introduction/aba_county_lawyers_intro.csv")

lawyers["lawyers_per_1000"] = (
    lawyers["Lawyers"] / lawyers["Pop2019"]
) * 1000

#number of counties, # with more than 10 per 1000 residents, last times 100

# Calculate lawyers per 1,000 residents
lawyers["lawyers_per_1000"] = (
    lawyers["Lawyers"] / lawyers["Pop2019"]
) * 1000


# 1. Counties below the lawyers threshold:
below_threshold_counties = lawyers.loc[
    lawyers["lawyers_per_1000"] < 1
].copy()

below_threshold_counties = below_threshold_counties.sort_values(
    "lawyers_per_1000",
    ascending=True,
)

# 2. Counties exceeding the lawyers threshold by a factor of 10: at least 10 lawyers per 1,000 residents
ten_times_threshold_counties = lawyers.loc[
    lawyers["lawyers_per_1000"] >= 10
].copy()

ten_times_threshold_counties = ten_times_threshold_counties.sort_values(
    "lawyers_per_1000",
    ascending=False,
)

# Calculate counts and percentages
total_counties = len(lawyers)

number_below_threshold = len(below_threshold_counties)
percent_below_threshold = (
    number_below_threshold / total_counties
) * 100

number_ten_times_threshold = len(ten_times_threshold_counties)
percent_ten_times_threshold = (
    number_ten_times_threshold / total_counties
) * 100


# Print the results
print(f"Total counties: {total_counties:,}")

print("\nLawyers threshold: fewer than 1 lawyer per 1,000 residents")
print(
    f"Counties below the threshold: "
    f"{number_below_threshold:,} of {total_counties:,}"
)
print(
    f"Percent below the threshold: "
    f"{percent_below_threshold:.2f}%"
)

print("\nTen times the lawyers threshold: at least 10 lawyers per 1,000")
print(
    f"Counties meeting this criterion: "
    f"{number_ten_times_threshold:,} of {total_counties:,}"
)
print(
    f"Percent meeting this criterion: "
    f"{percent_ten_times_threshold:.2f}%"
)

Total counties: 3,132

Lawyers threshold: fewer than 1 lawyer per 1,000 residents
Counties below the threshold: 1,286 of 3,132
Percent below the threshold: 41.06%

Ten times the lawyers threshold: at least 10 lawyers per 1,000
Counties meeting this criterion: 22 of 3,132
Percent meeting this criterion: 0.70%
